In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import glob
import geopandas as gpd
import geobr

In [2]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams['axes.grid'] = False
plt.rcParams['font.size'] = '14'
plt.rcParams["font.family"] = "Times New Roman"

In [3]:
# variables
threshold_pr_max = 450 #[mm]
threshold_pr_min = 0 #[mm]

# Data processing

In [4]:
df_data = pd.read_hdf('./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5', key = 'table_data', encoding = 'utf-8')
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
2908882,88690050,2023-12-27,0.00
2908883,88690050,2023-12-28,0.00
2908884,88690050,2023-12-29,2.00
2908885,88690050,2023-12-30,0.00


In [5]:
df_info = pd.read_hdf('./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024.h5', key = 'table_info', encoding = 'utf-8')
df_info = df_info[df_info['gauge_code'].isin(df_data['gauge_code'].unique())]
df_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


In [6]:
df_info_teste = df_info[df_info['gauge_code']=='350760501A']
df_info_teste

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
14776,350760501A,SÃO PAULO,BRAGANÇA PAULISTA,Lavapés,-22.9468,-46.5412,CEMADEN,CEMADEN,SP


In [7]:
gauge_counts = df_info['gauge_code'].value_counts()
print(gauge_counts)

gauge_code
00047000      1
251600301C    1
251700101C    1
251650802U    1
251620102A    1
             ..
02040006      1
02040007      1
02040008      1
02040009      1
S717          1
Name: count, Length: 18370, dtype: int64


In [8]:
df_complete_info = pd.merge(df_data, df_info, on='gauge_code', how = 'inner').sort_values('lat', ascending = True)
# .dropna(how='any')
df_complete_info

,gauge_code,datetime,rain_mm,state,city,name_station,lat,long,responsible,source,state_abbreviation
120617027,A899,2021-06-22,0.0,RIO GRANDE DO SUL,SANTA VITORIA DO PALMAR - BARRA DO CHUI,Santa Vitoria do Palmar - Barra do Chui | A899,-33.742222,-53.372222,INMET,INMET,RS
120617336,A899,2022-04-27,0.0,RIO GRANDE DO SUL,SANTA VITORIA DO PALMAR - BARRA DO CHUI,Santa Vitoria do Palmar - Barra do Chui | A899,-33.742222,-53.372222,INMET,INMET,RS
120617337,A899,2022-04-28,0.0,RIO GRANDE DO SUL,SANTA VITORIA DO PALMAR - BARRA DO CHUI,Santa Vitoria do Palmar - Barra do Chui | A899,-33.742222,-53.372222,INMET,INMET,RS
120617338,A899,2022-04-29,0.0,RIO GRANDE DO SUL,SANTA VITORIA DO PALMAR - BARRA DO CHUI,Santa Vitoria do Palmar - Barra do Chui | A899,-33.742222,-53.372222,INMET,INMET,RS
120617339,A899,2022-04-30,0.0,RIO GRANDE DO SUL,SANTA VITORIA DO PALMAR - BARRA DO CHUI,Santa Vitoria do Palmar - Barra do Chui | A899,-33.742222,-53.372222,INMET,INMET,RS
...,...,...,...,...,...,...,...,...,...,...,...
119574022,08460003,2006-04-29,5.8,RORAIMA,UIRAMUTA,ÁGUA FRIA,4.642800,-60.496400,ANA,HIDROWEB,RR
119574021,08460003,2006-04-28,0.0,RORAIMA,UIRAMUTA,ÁGUA FRIA,4.642800,-60.496400,ANA,HIDROWEB,RR
119574020,08460003,2006-04-27,1.4,RORAIMA,UIRAMUTA,ÁGUA FRIA,4.642800,-60.496400,ANA,HIDROWEB,RR
119574018,08460003,2006-04-25,9.1,RORAIMA,UIRAMUTA,ÁGUA FRIA,4.642800,-60.496400,ANA,HIDROWEB,RR


In [9]:
df_data_filtered = df_data[(df_data['rain_mm'] >= threshold_pr_min) & (df_data['rain_mm'] <= threshold_pr_max)]
df_info_filtered = df_info[df_info['gauge_code'].isin(df_data_filtered['gauge_code'].unique())]

In [10]:
print(len(df_complete_info) - len(df_data_filtered), 'data points were removed due to the precipitation threshold.')


-11356 data points were removed due to the precipitation threshold.


In [11]:
print((len(df_info) - len(df_info_filtered)), 'gauges were removed due to the precipitation threshold.')

0 gauges were removed due to the precipitation threshold.


In [12]:
print((len(df_data) - len(df_data_filtered))/len(df_data)*100,"%")

0.0041782101688251725 %


In [13]:
print((len(df_info) - len(df_info_filtered))/len(df_info)*100,"%")

0.0 %


In [14]:
df_data_filtered.to_hdf('./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5', key='table_data', mode='w', complevel=9, append = False, complib='blosc', encoding='utf-8')
df_info_filtered.to_hdf('./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5', key='table_info', mode='r+', complevel=9, append = False, complib='blosc', encoding='utf-8')

In [15]:
df_info_filtered = pd.read_hdf('./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5', key='table_info', encoding='utf-8')
df_info_filtered

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS
